In [3]:
# =========================
# Set up project path
# =========================

import sys
from pathlib import Path
from statsmodels.stats.proportion import proportions_ztest

# If this notebook is inside the notebooks/ folder
PROJECT_ROOT = Path.cwd().parent

# Add project root to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print("Current working directory:", Path.cwd())
print("Project root:", PROJECT_ROOT)
print("Project root exists:", PROJECT_ROOT.exists())

Current working directory: d:\2.DAP391m_Project\notebooks
Project root: d:\2.DAP391m_Project
Project root exists: True


In [4]:
# =========================
# Import Cleaned Dataset
# =========================

import pandas as pd
from src.config import CLEANED_DATA_PATH

print("Cleaned data path:", CLEANED_DATA_PATH)
print("File exists:", CLEANED_DATA_PATH.exists())

if not CLEANED_DATA_PATH.exists():
    raise FileNotFoundError(f"Cleaned dataset not found at: {CLEANED_DATA_PATH}")

df = pd.read_csv(CLEANED_DATA_PATH)

print("Cleaned dataset imported successfully!")
print("Shape:", df.shape)

display(df.head())

Cleaned data path: D:\2.DAP391m_Project\data\processed\dataset_cleaned.csv
File exists: True
Cleaned dataset imported successfully!
Shape: (288541, 9)


,user_id,timestamp,group,landing_page,converted,country,time_raw,elapsed_minutes,time_bucket
0,851104,11:48.6,control,old_page,0,US,11:48.6,11.810000,0-15_min
1,804228,01:45.2,control,old_page,0,US,01:45.2,1.753333,0-15_min
2,661590,55:06.2,treatment,new_page,0,US,55:06.2,55.103333,45-60_min
3,853541,28:03.1,treatment,new_page,0,US,28:03.1,28.051667,15-30_min
4,864975,52:26.2,control,old_page,1,US,52:26.2,52.436667,45-60_min


In [5]:
#Final Data Check 
print(df.info())
print(df.isnull().sum())
print(df.shape)
pd.crosstab(df["landing_page"], df["group"])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 288541 entries, 0 to 288540
Data columns (total 9 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   user_id          288541 non-null  int64  
 1   timestamp        288541 non-null  object 
 2   group            288541 non-null  object 
 3   landing_page     288541 non-null  object 
 4   converted        288541 non-null  int64  
 5   country          288541 non-null  object 
 6   time_raw         288541 non-null  object 
 7   elapsed_minutes  288541 non-null  float64
 8   time_bucket      288541 non-null  object 
dtypes: float64(1), int64(2), object(6)
memory usage: 19.8+ MB
None
user_id            0
timestamp          0
group              0
landing_page       0
converted          0
country            0
time_raw           0
elapsed_minutes    0
time_bucket        0
dtype: int64
(288541, 9)


group,control,treatment
landing_page,,
new_page,0,144315
old_page,144226,0


In [6]:
# Split new page and old page
old_page_data = df[df["landing_page"] == "old_page"]
new_page_data = df[df["landing_page"] == "new_page"]


In [7]:
#Calculate Users and Conversions
old_page_users = old_page_data["user_id"].count()
new_page_users = new_page_data["user_id"].count()

old_page_conversions = old_page_data["converted"].sum()

new_page_conversions = new_page_data["converted"].sum()


print("Old page users:", old_page_users)
print("Old page conversions:", old_page_conversions)

print("New page users:", new_page_users)
print("New page conversions:", new_page_conversions)

Old page users: 144226
Old page conversions: 17349
New page users: 144315
New page conversions: 17134


In [8]:
# Calculate Conversion Rate

old_page_conversion_rate = old_page_conversions / old_page_users
new_page_conversion_rate = new_page_conversions / new_page_users

old_page_conversion_rate_percent = old_page_conversion_rate * 100
new_page_conversion_rate_percent = new_page_conversion_rate * 100

print(f"Old page conversion rate: {old_page_conversion_rate_percent:.2f}%")
print(f"New page conversion rate: {new_page_conversion_rate_percent:.2f}%")

Old page conversion rate: 12.03%
New page conversion rate: 11.87%


In [9]:
conversion_summary = pd.DataFrame({
    "landing_page": ["old_page", "new_page"],
    "users": [old_page_users, new_page_users],
    "conversions": [old_page_conversions, new_page_conversions],
    "conversion_rate": [old_page_conversion_rate,new_page_conversion_rate],
    "conversion_rate_percent": [old_page_conversion_rate_percent,new_page_conversion_rate_percent]
})

conversion_summary

,landing_page,users,conversions,conversion_rate,conversion_rate_percent
0,old_page,144226,17349,0.120290,12.029038
1,new_page,144315,17134,0.118726,11.872640


In [10]:
# Calculate Absolute Lift and Relative Uplift

#Absolute Lift
absolute_lift_pp = (
    new_page_conversion_rate - old_page_conversion_rate
) * 100

print(f"Absolute lift: {absolute_lift_pp:.4f} percentage points")

# Relative Uplift
relative_uplift_percent = (
    (new_page_conversion_rate - old_page_conversion_rate)
    / old_page_conversion_rate
) * 100

print(f"Relative uplift: {relative_uplift_percent:.2f}%")

Absolute lift: -0.1564 percentage points
Relative uplift: -1.30%


In [11]:
lift_summary = pd.DataFrame({
    "old_cr": [old_page_conversion_rate],
    "new_cr": [new_page_conversion_rate],
    "old_cr_percent": [old_page_conversion_rate_percent],
    "new_cr_percent": [new_page_conversion_rate_percent],
    "absolute_lift_pp": [absolute_lift_pp],
    "relative_uplift_percent": [relative_uplift_percent]
})

lift_summary

,old_cr,new_cr,old_cr_percent,new_cr_percent,absolute_lift_pp,relative_uplift_percent
0,0.12029,0.118726,12.029038,11.87264,-0.156398,-1.300171


In [12]:
# Two-Proportion Z-test
count = [new_page_conversions, old_page_conversions]
nobs  = [new_page_users, old_page_users]

# Set up Two-Proportion Z-test
z_stat, p_value = proportions_ztest(
    count = count,
    nobs = nobs,
    alternative = "two-sided"
)

print(f"Z-statistic: {z_stat:.4f}")
print(f"P-value: {p_value:.4f}")

Z-statistic: -1.2949
P-value: 0.1953


In [13]:
alpha = 0.05
print("=======Conclusion=======")
if p_value < alpha:
    print("Decision: Reject H0")
    print("Interpretation: There is a statistically significant difference between old_page and new_page.")
else:
    print("Decision: Fail to reject H0")
    print("Interpretation: There is not enough evidence that old_page and new_page are different.")

=======Conclusion=======
Decision: Fail to reject H0
Interpretation: There is not enough evidence that old_page and new_page are different.


In [14]:
ztest_results = pd.DataFrame({
    "test_name": ["Two-Proportion Z-test"],
    "hypothesis_null": ["CR_new = CR_old"],
    "hypothesis_alternative": ["CR_new != CR_old"],
    "z_statistic": [z_stat],
    "p_value": [p_value],
    "alpha": [alpha],
    "decision": ["Reject H0" if p_value < alpha else "Fail to reject H0"],
    "interpretation": ["There is a statistically significant difference between old_page and new_page."
        if p_value < alpha else "There is not enough evidence that old_page and new_page are different."]
})

ztest_results

,test_name,hypothesis_null,hypothesis_alternative,z_statistic,p_value,alpha,decision,interpretation
0,Two-Proportion Z-test,CR_new = CR_old,CR_new != CR_old,-1.294922,0.195347,0.05,Fail to reject H0,There is not enough evidence that old_page and...


In [15]:
# =========================
# Save Result CSV Files
# =========================

from src.config import RESULTS_DIR
# Create results directory if it does not exist
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Save result tables
conversion_summary.to_csv(RESULTS_DIR / "conversion_summary.csv", index=False)
lift_summary.to_csv(RESULTS_DIR / "lift_summary.csv", index=False)
ztest_results.to_csv(RESULTS_DIR / "ztest_results.csv", index=False)

print("Result CSV files saved successfully!")
print("Saved to:", RESULTS_DIR)

Result CSV files saved successfully!
Saved to: D:\2.DAP391m_Project\data\results
